In [6]:
# ----- Import necessary packages -----
from datetime import datetime, date
from dateutil.tz import tzlocal
import os
import numpy as np
from uuid import uuid4
import pandas as pd

from pynwb import NWBHDF5IO, NWBFile, TimeSeries, read_nwb
from pynwb.file import Subject
from pynwb.behavior import (
    BehavioralEpochs,
    BehavioralEvents,
    BehavioralTimeSeries,
    CompassDirection,
    EyeTracking,
    Position,
    PupilTracking,
    SpatialSeries,
)
from pynwb.epoch import TimeIntervals
from pynwb.image import ImageSeries
import wave
from pynwb import get_class, load_namespaces

from ndx_manoli_meta import AssayMetadata, IndividualMetadata 

In [88]:
edgecol = colors.loc[(meta.FocalSex,meta.FocalGT),('Edge',meta.StimType)]
facecol = colors.loc[(meta.FocalSex,meta.FocalGT),('Face',meta.StimType)]
np.array([edgecol,facecol]).transpose(), # colors to use for plotting (3 floats), doesnt work for two sets of colors


(array([[0, 0],
        [0, 0],
        [0, 0]]),)

In [81]:
colors = pd.DataFrame(columns = pd.MultiIndex.from_product([['Edge', 'Face'], ['Partner','Stranger','Isolation']], 
                                                           names=['ColorType', 'Stimulus']),
                      index = pd.MultiIndex.from_product([['Female','Male'], ['WT', 'HO']], 
                                                         names=['Sex', 'Genotype']))
for idx in colors.index:
    for col in colors.columns:
        if idx[1]=='HO' and col[0]=='Face': c = [None,None,None]
        else: 
            if idx[0]=='Female':
                if col[1]=='Partner': c = [0,0,0]
                elif col[1]=='Stranger': c = [0,0,0]
                elif col[1]=='Isolation': c = [0,0,0]
            elif idx[0]=='Male':
                if col[1]=='Partner': c = [0,0,0]
                elif col[1]=='Stranger': c = [0,0,0]
                elif col[1]=='Isolation': c = [0,0,0]
        colors.loc[idx,col] = c
colors

ColorType             Edge                                      Face  \
Stimulus           Partner   Stranger  Isolation             Partner   
Sex    Genotype                                                        
Female WT        [0, 0, 0]  [0, 0, 0]  [0, 0, 0]           [0, 0, 0]   
       HO        [0, 0, 0]  [0, 0, 0]  [0, 0, 0]  [None, None, None]   
Male   WT        [0, 0, 0]  [0, 0, 0]  [0, 0, 0]           [0, 0, 0]   
       HO        [0, 0, 0]  [0, 0, 0]  [0, 0, 0]  [None, None, None]   

ColorType                                                
Stimulus                   Stranger           Isolation  
Sex    Genotype                                          
Female WT                 [0, 0, 0]           [0, 0, 0]  
       HO        [None, None, None]  [None, None, None]  
Male   WT                 [0, 0, 0]           [0, 0, 0]  
       HO        [None, None, None]  [None, None, None]

In [7]:
def import_aggregated(aggregated_events):
    all_events = pd.read_excel(aggregated_events, usecols = 'A,G:I,K:N')
    all_events = all_events.sort_values('Observation id',ignore_index=True)
    all_events.columns = [x.lower().replace(' ','_').replace('(','').replace(')','') for x in all_events.columns]
    all_events = all_events.sort_values(by=['observation_id','subject','start_s'],ascending=[True,False,True],ignore_index=True)
    return all_events

def import_boris_metadata(aggregated_events):
    boris_meta = pd.read_excel(aggregated_events, usecols = 'A,E:F').drop_duplicates()
    boris_meta.index = [p.split('_')[0] for p in boris_meta['Observation id']]
    boris_meta = boris_meta.sort_index()
    return boris_meta

def recording_date_info(meta):
    # calculate days post pairing
    rdate = date(int(str(meta.RecDate)[0:4]),int(str(meta.RecDate)[4:6]),int(str(meta.RecDate)[6:]))
    pdate = date(int(str(meta.PairDate)[0:4]),int(str(meta.PairDate)[4:6]),int(str(meta.PairDate)[6:]))
    dpp = rdate-pdate # days post-pairing
    return rdate, dpp

def recording_time_info(meta):
    recDate, dpp = recording_date_info(meta) # find session date
    if type(meta.VideoFile)==str: 
        # Extract time from video name
        video_file = meta.VideoFile
        timepieces = video_file.split('.')[0].split('_')[-1].split('-')
        sess_start = datetime(year=recDate.year,month=recDate.month,day=recDate.day,
                              hour=int(timepieces[0]),minute=int(timepieces[1]),second=int(timepieces[2]),
                              tzinfo = tzlocal())
    else:
        sess_start = datetime(year=rdate.year,month=rdate.month,day=rdate.day,
                              hour=int(str(meta.RecTime)[0:2]),minute=int(str(meta.RecTime)[2:]),
                              tzinfo = tzlocal())
    return sess_start

def isolation_duration(meta):
    if np.sum([np.isnan(v) for v in meta['IsolationTime':'RecTime']]) == 0: # are there values listed for isolation and recording times?
        isoDur_HM = [float(rec)-float(iso) for rec,iso in zip([str(meta.RecTime)[0:2],str(meta.RecTime)[2:]],[str(meta.IsolationTime)[0:2],str(meta.IsolationTime)[2:]])]
        iso_Sec = (3600*isoDur_HM[0])+(60*isoDur_HM[1])
    else: iso_Sec = None
    return iso_Sec

def make_assay_metadata(meta,assay_type,b):

    recDate, dpp = recording_date_info(meta)
    
    if assay_type=='interaction': assaydescr = 'pre-perfusion isolation period'
    elif assay_type=='isolation': assaydescr = 'brief interaction'
    
    isoSec = isolation_duration(meta)

    if assay_type=='interaction': 
        desc = 'Brief interaction with a stimulus animal or isolation-only control.'
        
        if type(b)==type(None):
            dur = float(meta.VideoAssayDur)
        else:
            dur = float(b['Total length'])
            
        
        metaObj = AssayMetadata(
            ######################### Basic Info #########################
            assay_type = assay_type, # what assay type (REQUIRED)
            assay_type__description = desc, # description of assay
            assay_type__days_post_pairing = dpp.days, # days post pairing (int)
            duration = dur, # assay length in seconds (float)
            room = meta.AssayRoom, # room
            timeline = meta.Timeline, # name timeline of assays
            genotype_confirmed = meta.FocalGT == meta.FocalGTconfirmed, # boolean, did genotype get confirmed
            exclude_flag = not(meta.DataComplete), # exclude assay or not, boolean
            assay_type__focal_ETside = meta.FocalET, # focal ear tag side
            assay_type__HomeCage_Box = str(meta.Box), # home cage box label
            #assay_type__annotations = 'annos' # filename if separate events file
            ######################### Stimulus Info #########################
            assay_type__stim_type = meta.StimType, # stimulus animal type
            assay_type__stim_ID = meta.StimID, # stim ear tag number
            assay_type__stim_GT = meta.StimGT, # stim genotype
            assay_type__stim_sex = meta.StimSex, # stim sex
            assay_type__stim_DOB = str(meta.StimDOB), # stim DOB
            assay_type__stim_ETside = meta.StimET, # stim ear tag side
            assay_type__stim_fam = meta.StimFam, # stim family
            ######################### BORIS and Scoring #########################
            ethogram = str(meta.Ethogram), # name of ethogram used for scoring
            scorer = str(meta.ScoredBy), # person who scored assay
            ######################### Miscellaneous #########################
            assay_type__divided = False, # boolean, did assay occur with a divider
            assay_type__isolation_length = int(isoSec), # time in sec of focal animal isolation (int)
            #colors = [colors.loc[(meta.FocalSex,meta.FocalGT),('Edge',meta.StimType)], 
            #    colors.loc[(meta.FocalSex,meta.FocalGT),('Face',meta.StimType)]], # colors to use for plotting (3 floats), doesnt work for two sets of colors
            camera_type = 'ELP USB Camera (ELP-USBFHD01M-L180)', # type of camera used for recording
            # path_to_histology_files = 'path_to_histology_files', # path to any histology files
            timeline_complete = meta.DataComplete # boolean, does animal have all assays/data methods complete
        )
    elif assay_type=='isolation': 
        desc = 'Approximately 85-minute isolation period after interaction but before perfusion.'
        metaObj = AssayMetadata(
            ######################### Basic Info #########################
            assay_type = assay_type, # what assay type (REQUIRED)
            assay_type__description = desc, # description of assay
            assay_type__days_post_pairing = dpp.days, # days post pairing (int)
            duration = float(meta.VideoAssayDur), # assay length in seconds (float)
            room = meta.AssayRoom, # room
            timeline = meta.Timeline, # name timeline of assays
            genotype_confirmed = meta.FocalGT == meta.FocalGTconfirmed, # boolean, did genotype get confirmed
            exclude_flag = not(meta.DataComplete), # exclude assay or not, boolean
            #assay_type__annotations = 'annos' # filename if separate events file
            assay_type__focal_ETside = meta.FocalET, # focal ear tag side
            assay_type__HomeCage_Box = str(meta.Box), # home cage box label
            ######################### BORIS and Scoring #########################
            #ethogram = meta.Ethogram, # name of ethogram used for scoring
            #scorer = meta.ScoredBy, # person who scored assay
            ######################### Miscellaneous #########################
            assay_type__divided = False, # boolean, did assay occur with a divider
            #assay_type__isolation_length = int(iso_Sec), # time in sec of focal animal isolation (int)
            #colors = [colors.loc[(meta.FocalSex,meta.FocalGT),('Edge',meta.StimType)], 
            #    colors.loc[(meta.FocalSex,meta.FocalGT),('Face',meta.StimType)]], # colors to use for plotting (3 floats), doesnt work for two sets of colors
            camera_type = 'ELP USB Camera (ELP-USBFHD01M-L180)', # type of camera used for recording
            # path_to_histology_files = 'path_to_histology_files', # path to any histology files
            timeline_complete = meta.DataComplete # boolean, does animal have all assays/data methods complete
        )
    return metaObj, assaydescr

def make_NWB_file(meta, ptag, assaydescr, nwbfilename): 
    session_start = recording_time_info(meta) # find session start time
    session_description = f'Behavioral annotations from {meta.FocalID} ({ptag}, {meta.FocalSex}, {meta.FocalGT}) during {assaydescr} [cohort: {meta.CohortTag}]'
    nwbfile = NWBFile(
        session_description = session_description,
        identifier = str(uuid4()),
        session_start_time = session_start,
        lab="Manoli @ UCSF",
        experimenter=meta.RanBy,
        session_id = nwbfilename[0:-4]
    )
    return nwbfile

def make_subject_metadata(meta):
    if str(meta.FocalSex).endswith('ale'): sexlabel = meta.FocalSex
    else:
        if meta.FocalSex=='M': sexlabel = 'Male'
        elif meta.FocalSex=='F': sexlabel = 'Female'
    focalDOB = meta[sexlabel+'DOB']
    subjectObj = Subject(
        subject_id = meta.FocalID,
        species = 'Microtus ochrogaster',
        sex = meta.FocalSex,
        genotype = meta.FocalGT,
        date_of_birth = datetime(year=int(str(focalDOB)[0:4]),month=int(str(focalDOB)[4:6]),day=int(str(focalDOB)[6:]),tzinfo = tzlocal()),
        description = meta[sexlabel+'ID']+': '+meta[sexlabel+'GT']+', '+meta[sexlabel+'Fam']+', '+str(meta[sexlabel+'DOB'])
    )
    return subjectObj

def make_partner_metadata(meta):
    if str(meta.FocalSex).upper().startswith('F'): partner_sex = 'Male'
    elif str(meta.FocalSex).upper().startswith('M'): partner_sex = 'Female'
    partnerDOB = str(meta[partner_sex+'DOB'])
    partnerObj = IndividualMetadata(
        genotype = meta[partner_sex+'GT'],
        species = 'Microtus ochrogaster',
        sex = partner_sex,
        individual_id = meta[partner_sex+'ID'],
        description = meta[partner_sex+'ID']+': '+meta[partner_sex+'GT']+', '+meta[partner_sex+'Fam']+', '+str(meta[partner_sex+'DOB']),
        date_of_birth = datetime(year=int(partnerDOB[0:4]),month=int(partnerDOB[4:6]),day=int(partnerDOB[6:]),tzinfo = tzlocal()).isoformat(),
        age__reference = 'birth'
        )
    return partnerObj

def make_video_metadata(ptag, meta, b):
    if type(b)==type(None):
        video_ext_file = ImageSeries(
            name='behaviorVideo',
            description='Raw original video (estimated FPS).',
            unit='n.a.',
            external_file=[os.path.join(meta.VideoPath,meta.VideoFile)],
            format='external',
            starting_time=float(meta.VideoAssayStart),
            rate=float(60),
        )
    else:
        video_ext_file = ImageSeries(
            name='behaviorVideo',
            description='Raw original video.',
            unit='n.a.',
            external_file=[os.path.join(meta.VideoPath,meta.VideoFile)],
            format='external',
            starting_time=float(meta.VideoAssayStart),
            rate=float(b['FPS']),
        )
    return video_ext_file

def make_audio_metadata(meta, project_path = '/Users/joshsteighner/Library/CloudStorage/Box-Box/30s_cfos'):
    aud_path = (meta.AudioPath+meta.AudioFile).replace('..',project_path)
    with wave.open(aud_path, "rb") as wave_file: # find sample rate
        sampling_rate = wave_file.getframerate()
    Fs = float(sampling_rate)
    aud_ext_file = ImageSeries( 
        name='behaviorAudio',
        description='Raw freefield audio',
        unit='n.a.',
        external_file=[meta.AudioPath+meta.AudioFile],
        format='external',
        starting_time=meta.AudioAssayStart,
        rate=Fs,
        offset=meta.estAVOffset,
        comments='offset = audio_start_time – video_start_time, in seconds',
    )
    return aud_ext_file

def make_behavior_intervals(aggregated, ptag, individual):
    filtered_events = aggregated.loc[[i for i in aggregated.index if (aggregated.loc[i,'observation_id'].startswith(ptag)) and (aggregated.loc[i,'subject']==individual.title())]].sort_values(by='start_s').reset_index(drop=True)
    filtered_events.head()
    # make NWB object corresponding to the annotation table
    behavior_intervals = TimeIntervals(name=individual.lower()+'_behavior',
        description='Intervals of scored behavior of '+individual+' animal from '+ptag+'.')
    behavior_intervals.add_column(name="behavior", description="The annotation from the ethogram.")
    behavior_intervals.add_column(name="duration", description="Duration of the behavior.")
    behavior_intervals.add_column(name="atype", description="Point or state event.")
    # populate table
    for i,row in filtered_events.iterrows(): behavior_intervals.add_row(start_time=row.start_s,
                                                                        stop_time=row.stop_s,
                                                                        behavior=row.behavior,
                                                                        atype=row.behavior_type,
                                                                        duration=row.duration_s
                                                                       )
    return behavior_intervals

def generate_NWB_file(my_row,all_events,boris_meta,nwbfile_path):

    (ptag, assay_type), meta = my_row
    nwbfilename = f'{ptag}_{assay_type}.nwb'
    
    wfullpath = os.path.join(nwbfile_path,nwbfilename)
    if not os.path.exists(wfullpath):
    
        write_NWB_to_disk = not(pd.isna(meta.ScoredBy))
    
        print('Generating: '+'/'.join(wfullpath.split('/')[-3:])+' ...')
    
        if ptag in [obsID.split('_')[0] for obsID in all_events.observation_id.unique()]:
            b = boris_meta.loc[ptag]
        else: b = None
    
        metaObj, assaydescr = make_assay_metadata(meta,assay_type,b) # make assay metadata object
        nwbfile = make_NWB_file(meta, ptag, assaydescr, nwbfilename) # make NWB file
        
        nwbfile.add_lab_meta_data(lab_meta_data=metaObj) # add assay metadata
    
        print('... adding subject and partner metadata ...')
        nwbfile.subject = make_subject_metadata(meta) # make and add subject metadata
        partner_data = make_partner_metadata(meta)
        nwbfile.add_lab_meta_data(lab_meta_data=partner_data) # make and add partner metadata
    
        if type(meta.VideoFile)==str: 
            nwbfile.add_acquisition(make_video_metadata(ptag, meta, b)) # make and add video metadata, if it exists
            print('... adding video metadata ...')
    
        if type(meta.AudioFile)==str:
            nwbfile.add_acquisition(make_audio_metadata(meta)) # make and add audio metadata, if it exists
            print('... adding audio metadata ...')
    
        if (type(meta.ScoredBy)==str) and (assay_type.lower().startswith('int')) and (ptag in [obsID.split('_')[0] for obsID in all_events.observation_id.unique()]):
            print('... adding behavior data ...')
            for individual in ['subject','stimulus']: 
                nwbfile.add_time_intervals(make_behavior_intervals(all_events, ptag,individual))
    
        if write_NWB_to_disk:
            print('... saving file ...')
            with NWBHDF5IO(wfullpath, "w") as io:
                io.write(nwbfile)
        print('...done.')
    
    else:
        print('/'.join(wfullpath.split('/')[-3:])+' already exists. Skipping ...')
        nwbfile = None

    return nwbfile 



In [9]:
############################################## Set Paths ##############################################
aggregated_path = '/Users/joshsteighner/Library/CloudStorage/Box-Box/30s_cfos/BORIS/aggregated_events_updated.xlsx'
metadata_path = '/Users/joshsteighner/Library/CloudStorage/Box-Box/30s_cfos/NWB_ShortInteraction_Metadata.xlsx'
nwbfile_path = '/Users/joshsteighner/Library/CloudStorage/Box-Box/30s_cfos/NWB/'

# Import tables
metadata = pd.read_excel(metadata_path, usecols = 'A:AX',index_col=[0,11]) # Import Metadata Excel
all_events = import_aggregated(aggregated_path) # Import aggregated events sheet
boris_meta = import_boris_metadata(aggregated_path)

# Iterate over all of the rows in the metadata tables
for row in metadata.iterrows(): nwbfile = generate_NWB_file(row,all_events,boris_meta,nwbfile_path)    


30s_cfos/NWB/Pair21_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair22_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair23_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair24_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair25_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair26_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair27_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair28_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair29_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair30_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair31_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair32_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair33_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair34_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair35_interaction.nwb already exists. Skipping ...
30s_cfos/NWB/Pair36_inter

In [5]:
filepath = "/Users/joshsteighner/Library/CloudStorage/Box-Box/30s_cfos/NWB/Pair39_interaction.nwb"
from pynwb import read_nwb

nwbfile = read_nwb(filepath)
nwbfile

Data type,uint8
Shape,"(0, 0, 0)"
Array size,0.00 bytes
Chunk shape,None
Compression,None
Compression opts,None
Compression ratio,undefined
Data type,object
Shape,"(1,)"
Array size,8.00 bytes
Chunk shape,None


In [3]:
AssayMetadata(
    ######################### Basic Info #########################
    assay_type = 'test1', # what assay type (REQUIRED)
    assay_type__description = 'desc', # description of assay
    exclude_flag = False,
    duration=float(1),
    colors =  [[0,0,0],[0,0,0]],
)

vole_metadata abc.AssayMetadata at 0x5219746288
Fields:
  assay_type: test1
  assay_type__description: desc
  colors: [[0 0 0]
 [0 0 0]]
  duration: 1.0
  exclude_flag: False